# Beste praksis i data science

I de foregående kapitlene har vi lært om de ulike tekniske delene av data science. Men i virkeligheten er data science et lagarbeid som foregår over tid, i komplekse organisasjoner og med virkelige konsekvenser. Gode tekniske ferdigheter er nødvendige, men ikke tilstrekkelige.

I dette kapittelet ser vi på beste praksis: de vanene, metodene og prinsippene som skiller robuste, pålitelige data science-prosjekter fra kaotiske og ureproduserbare.

## Reproduserbarhet

Hva er reproduserbarhet? En analyse er reproduserbar hvis noen annen (eller du selv senere), med de samme dataene og de samme metodene, kommer frem til de samme resultatene. Dette er et grunnleggende krav i vitenskap, men det er overraskende vanskelig å oppnå i praksis. 

Her går vi gjennom noen prinsipper og vaner som fører til bedre reproduserbarhet.

**Rådataene er hellige**. Vi endrer aldri rådataene (*raw data*) våre direkte. All opprydding, filtrering og omkoding skjer i kode, slik at vi alltid kan gå tilbake til utgangspunktet. Hvis vi retter en feil for hånd i en CSV-fil, finnes det ingen oppskrift på hva vi gjorde og da er resultatet ikke lenger reproduserbart.

**Bruk en fast frøverdi** (*seed*). Mange metoder bruker tilfeldighet. For eksempel oppdelingen i trenings-, validerings- og testdata eller initialisering av en modell. Med en fast frøverdi får vi det samme "tilfeldige" resultatet hver gang. Du har allerede sett `random_state=0` i `train_test_split` flere ganger. Det er nettopp dette i praksis.

In [1]:
import numpy as np

# Uten fast seed: ulikt resultat hver gang vi kjører
print("Uten seed:", np.random.default_rng().random(3))
print("Uten seed:", np.random.default_rng().random(3))

# Med fast seed: alltid det samme resultatet
print("Med seed :", np.random.default_rng(42).random(3))
print("Med seed :", np.random.default_rng(42).random(3))

Uten seed: [0.58451649 0.2655799  0.29864764]
Uten seed: [0.5661934  0.45293505 0.08237357]
Med seed : [0.77395605 0.43887844 0.85859792]
Med seed : [0.77395605 0.43887844 0.85859792]


Det er god praksis å definere frøverdien som en konstant øverst i notebooken eller i en konfigurasjonsfil, slik at man enkelt kan endre den og sjekke at resultatene ikke er sensitive til den spesifikke verdien.

**Kjør alt på nytt, ovenfra og ned**. Jupyter-notebooks har en lumsk felle: du kan kjøre celler i hvilken som helst rekkefølge, og en variabel kan ligge igjen i minnet fra en celle du senere har slettet eller endret. Da kan notebooken se ut til å virke hos deg, men feile for alle andre. Før du deler en notebook, bruk derfor altid **«Restart Kernel and Run All»**. Hvis notebooken kjører rent ovenfra og ned, er rekkefølgen riktig og ingen skjult tilstand lurer i bakgrunne.

**Bruk relative stier.** Skriv `data/strompris.csv`, ikke
`C:/Users/pekka/data/strompris.csv`. Absolutte stier virker bare på din
maskin. Med relative stier kan hvem som helst klone prosjektet og kjøre det.

**Dokumenter miljøet ditt.** Koden din avhenger av hvilke pakker (og hvilke
versjoner) du har installert. Lagre disse i en `requirements.txt`-fil, gjerne i et
eget virtuelt miljø (*virtual environment*), slik at andre kan installere akkurat
det samme:

```text
# requirements.txt
pandas==2.2.2
scikit-learn==1.5.0
matplotlib==3.9.0
```

```bash
# installere alt på nytt i et nytt miljø
pip install -r requirements.txt
```

**Generer rapporter automatisk.** Som nevnt i kapittelet om datakommunikasjon: jo
mer av figurer og tall som genereres direkte fra koden, jo mindre er sjansen for
at rapporten viser en utdatert versjon etter at du har endret noe i
dataforberedelsen.

## Versjonskontroll med Git

Har du noen gang hatt filer som heter `analyse.py`, `analyse_v2.py`,
`analyse_endelig.py` og `analyse_endelig_NÅ_på_ekte.py`? Versjonskontroll
(*version control*) løser nettopp dette problemet. Git er det desidert mest
brukte verktøyet, og det er verdt å lære seg tidlig.

Et Git-repositorium (*repository*, ofte kalt «repo») er en mappe der Git
holder oversikt over alle endringer i filene over tid. De viktigste begrepene:

- En `commit` er et øyeblikksbilde av prosjektet med en kort beskrivelse av hva
  du endret. Du kan alltid gå tilbake til en tidligere commit.
- Med `branch` og `merge` kan du jobbe på en idé uten å ødelegge den
  fungerende versjonen, og flette den inn igjen når den er klar.
- Med en tjeneste som GitHub eller GitLab kan du dele repoet og samarbeide
  med andre.

En typisk arbeidsflyt ser slik ut:

```bash
git init                      # opprett et repo i mappen
git add analyse.py            # marker filer som skal med
git commit -m "Legg til EDA av strømpris"   # lagre et øyeblikksbilde
git push                      # send til GitHub/GitLab
```

**Skriv commit-meldinger som forklarer *hvorfor*.** "Fikset bug" sier lite;
"Rett feil i imputering: brukte middelverdi fra hele datasettet i stedet for kun
treningsdata" forteller den neste leseren (ofte deg selv) hva som faktisk
skjedde.

**Ikke sjekk inn alt.** Rådata, store filer, hemmeligheter (passord, API-nøkler)
og automatisk genererte filer bør ikke ligge i repoet. Det styrer vi med en
`.gitignore`-fil:

```text
# .gitignore
data/raw/          # store rådatafiler
*.pkl              # lagrede modeller
.env               # API-nøkler og passord
__pycache__/
.ipynb_checkpoints/
```

> **Merk:** API-nøkler og passord skal *aldri* ligge i koden eller i repoet. Et
> vanlig mønster er å legge dem i en `.env`-fil som er listet i `.gitignore`.

## Kodestruktur

Kode leses langt oftere enn den skrives. Litt struktur gjør koden lettere å
forstå, lettere å feilsøke og mindre utsatt for feil.

**Skriv funksjoner i stedet for å kopiere og lime.** Hvis du gjør den samme
beregningen flere ganger, lag en funksjon. Da finnes logikken ett sted: retter du
en feil, retter du den overalt. Dette prinsippet kalles ofte *DRY* – «Don't
Repeat Yourself».

In [ ]:
import pandas as pd

df = pd.DataFrame({"temp": [4.0, 8.0, 12.0], "vind": [2.0, 5.0, 9.0]})

# FØR: samme beregning kopiert for hver kolonne – lett å gjøre en skrivefeil
df["temp_norm"] = (df["temp"] - df["temp"].mean()) / df["temp"].std()
df["vind_norm"] = (df["vind"] - df["vind"].mean()) / df["vind"].std()
df

In [ ]:
# ETTER: én funksjon, brukt for hver kolonne
def standardiser(kolonne):
    'Trekk fra middelverdien og del på standardavviket.'
    return (kolonne - kolonne.mean()) / kolonne.std()

for navn in ["temp", "vind"]:
    df[f"{navn}_norm"] = standardiser(df[navn])
df

> **Et viktig forbehold:** her regner vi ut middelverdi og standardavvik fra
> *hele* datasettet. I en ekte modell ville dette være datalekkasje
> (*data leakage*) – vi skal alltid tilpasse slike transformasjoner kun på
> treningsdata, akkurat som vi gjorde med imputering. (I praksis bruker vi gjerne
> `StandardScaler` fra `scikit-learn` nettopp fordi den skiller `fit` på
> treningsdata fra `transform` på validerings- og testdata.)

**Gi ting beskrivende navn.** `gjennomsnittstemperatur` er bedre enn `g`, `x2`
eller `tmp`. Den lille ekstra skrivingen sparer mye gjetting senere.

**Kommentarer skal forklare *hvorfor*, ikke *hva*.** Koden viser allerede hva som
skjer. En god kommentar forteller hvorfor du gjorde et valg:

```python
# Vi filtrerer bort negative priser fordi de skyldes en kjent målefeil i 2021
df = df[df["pris"] >= 0]
```

**Notebook eller modul?** Notebooks er gode til utforsking, fortelling og
rapporter. Men når koden vokser, eller når den samme funksjonen skal brukes flere
steder, er det ryddigere å flytte den ut i en vanlig Python-fil (en *modul*,
f.eks. `forberedelse.py`) og importere den. En vanlig arbeidsdeling er:
*funksjoner i moduler, fortellingen i notebooken.*

## Datahåndtering

**Skill mellom rå og bearbeidet data.** En velprøvd mappestruktur ser slik ut:

```text
prosjekt/
├── data/
│   ├── raw/          # rådata, røres aldri
│   └── processed/    # data etter opprydding, generert av kode
├── notebooks/        # utforsking og rapporter
├── src/              # gjenbrukbare funksjoner (moduler)
├── requirements.txt
└── README.md         # hva er dette prosjektet, og hvordan kjøres det?
```

Poenget er at alt i `processed/` skal kunne *gjenskapes* fra `raw/` ved å kjøre
koden. Da kan du trygt slette `processed/` og lage det på nytt – og du vet at det
alltid er i synk med koden din.

**Skriv en datakatalog (*data dictionary*).** En kort beskrivelse av hver variabel
– navn, enhet, mulige verdier, og hva «manglende» betyr – er gull verdt. Husk
diskusjonen om manglende data: betyr en tom celle «null», «vet ikke», eller «ikke
målt»? Det bør stå et sted.

**Dokumenter hvor dataene kommer fra.** Kilde, nedlastingsdato og lisens. Data endrer seg over
tid, så en analyse uten dato er vanskelig å etterprøve.

## Enkle tester og fornuftsjekker

I programvareutvikling skriver man formelle tester. I data science holder det
ofte langt å legge inn enkle **fornuftssjekker** (*sanity checks*) – små
påstander om at dataene og resultatene ser fornuftige ut. En `assert`-setning
stopper koden umiddelbart hvis noe er galt, slik at en feil ikke får forplante seg
stille gjennom hele analysen.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "alder": [34, 52, 19, 67, 41],
    "kjonn": ["K", "M", "M", "K", "M"],
})

# Fornuftssjekker: stopp tidlig hvis noe er åpenbart galt
assert df["alder"].between(0, 120).all(), "Urimelige aldersverdier!"
assert df["kjonn"].isin(["K", "M"]).all(), "Uventede verdier i kjonn!"
assert df.notna().all().all(), "Det finnes manglende verdier!"

print("Alle fornuftssjekker passerte.")

Prøv gjerne å endre `34` til `200` i cellen over og kjøre på nytt – da ser du at
sjekken fanger opp problemet umiddelbart, i stedet for at den urimelige verdien
sniker seg inn i modellen.

Nyttige ting å sjekke underveis:

- **Form og størrelse:** har datarammen det antallet rader og kolonner du
  forventer etter en sammenslåing eller filtrering?
- **Verdiområder:** er priser ikke-negative, sannsynligheter mellom 0 og 1,
  datoer innenfor perioden du studerer?
- **Ingen datalekkasje:** brukte du bare treningsdata til å tilpasse imputering,
  skalering og modell?
- **Sammenlign med en grunnlinjemodell (*baseline*):** er den avanserte modellen
  faktisk bedre enn å bare gjette middelverdien eller den vanligste klassen? Hvis
  ikke, er det ofte et tegn på en feil.

## Etikk og ansvar

Beste praksis handler ikke bare om kode, men også om konsekvensene av arbeidet
vårt. Vi går grundig gjennom dette i kapittelet om studiedesign, så her nøyer vi
oss med en påminnelse: tenk gjennom personvern og GDPR når du jobber med
personopplysninger, vær oppmerksom på skjevhet (*bias*) i data og modeller, og vær
åpen om begrensningene i analysen din. Transparens, at andre kan se hva du har
gjort og hvorfor, er like mye en etisk dyd som en teknisk en, og det er nettopp
det reproduserbarhet og god dokumentasjon gir oss.